# Semantic Clustering using BERTopic
This notebook demonstrates step-by-step how to perform semantic clustering on your dataset based on the specific columns requested.

### Step 1: Install Dependencies
First, make sure you have all the required libraries installed.

In [ ]:
!pip install pandas bertopic sentence-transformers

### Step 2: Import Libraries and Load Data
Here we import the libraries and load the dataset. We also replace any `NaN` values with empty strings to prevent errors during text concatenation.

In [ ]:
import pandas as pd
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

# ---------------------------------------------------------
# Load your dataset (Replace 'your_dataset.csv' with your actual file path)
# df = pd.read_csv('your_dataset.csv')
# ---------------------------------------------------------

# For the sake of this notebook running without errors if you test it right away, 
# here is a tiny placeholder dataset. 
# REMOVE the dummy df below and UNCOMMENT the pd.read_csv line above to use your real data.
df = pd.DataFrame({
    'Norm_Section': ['Sec 1', 'Sec 2', 'Sec 1'],
    'Section_header_commented': ['Header A', 'Header B', 'Header A'],
    'Category_commented': ['Cat 1', 'Cat 2', 'Cat 1'],
    'Original (removed=red strike)': ['Some original text removed', 'Other text', 'More removed text'],
    'Text Portion Deleted': ['Deleted text snippet', 'Another deleted', 'Snippet 3']
})

# Handle missing values by replacing NaNs with empty strings
df = df.fillna('')
df.head()

### Step 3: Initialize the Model
We initialize the specific sentence transformer model requested: `all-MiniLM-L12-v2`.

In [ ]:
# Define the model as requested
embedding_model = SentenceTransformer("all-MiniLM-L12-v2")

# Initialize BERTopic with the specified embedding model
topic_model = BERTopic(embedding_model=embedding_model)

### Step 4: Task 1 - Cluster based on Columns 2, 3, 4, and 12
Columns:
1. `Norm_Section`
2. `Section_header_commented`
3. `Category_commented`
4. `Original (removed=red strike)`

In [ ]:
print("Starting Task 1...")

# Combine the text from the specified columns
df['combined_text_task1'] = (
    df['Norm_Section'].astype(str) + " " +
    df['Section_header_commented'].astype(str) + " " +
    df['Category_commented'].astype(str) + " " +
    df['Original (removed=red strike)'].astype(str)
)

# Convert to list for BERTopic
docs_task1 = df['combined_text_task1'].tolist()

# Fit and transform to generate topics/clusters
# Note: For very small dummy datasets, BERTopic might return -1 (outlier) for everything. 
# It will work properly on your real, larger dataset.
topics_task1, probabilities_task1 = topic_model.fit_transform(docs_task1)

# Assign the cluster labels back to the dataframe
df['Cluster_Task1'] = topics_task1

# View the generated topics
topic_model.get_topic_info()

### Step 5: Task 2 - Cluster based on Columns 2, 3, 4, and 15
Columns:
1. `Norm_Section`
2. `Section_header_commented`
3. `Category_commented`
4. `Text Portion Deleted`

In [ ]:
print("Starting Task 2...")

# Combine the text from the specified columns (swapping col 12 for 15)
df['combined_text_task2'] = (
    df['Norm_Section'].astype(str) + " " +
    df['Section_header_commented'].astype(str) + " " +
    df['Category_commented'].astype(str) + " " +
    df['Text Portion Deleted'].astype(str)
)

# Convert to list for BERTopic
docs_task2 = df['combined_text_task2'].tolist()

# Re-fit the model on the new document set
topics_task2, probabilities_task2 = topic_model.fit_transform(docs_task2)

# Assign the cluster labels back to the dataframe
df['Cluster_Task2'] = topics_task2

# View the generated topics
topic_model.get_topic_info()

### Step 6: Save Results
Finally, we save the clustered data to a new CSV file.

In [ ]:
# Save the Results
df.to_csv('clustered_dataset.csv', index=False)
print("Results saved successfully to 'clustered_dataset.csv'.")
df.head()